# 🔐 File Integrity Checker

**Project:** Verify the integrity of log files to detect tampering using SHA-256 hashing.

**How it works:**
1. A cryptographic hash (SHA-256) is computed for each log file — a unique 64-character fingerprint
2. Hashes are stored in `hashes.json` as the trusted baseline
3. On future runs, new hashes are compared against the baseline
4. Any mismatch = file was tampered with

---

## 1. Imports

All libraries used here come built-in with Python — no `pip install` needed.

| Library | What it does |
|---------|-------------|
| `hashlib` | Provides SHA-256 and other hash functions |
| `os` | File and folder operations |
| `json` | Save/load the hash database |
| `sys` | Read command-line arguments |
| `datetime` | Record timestamps |

In [ ]:
# ============================================
# Cell 1 — Imports
# ============================================
import hashlib
import os
import json
import sys
from datetime import datetime

print("✅ All imports successful")

## 2. Configuration

`HASH_DB` is the filename where all fingerprints are stored.
Keeping it as a constant means you only need to change it in one place if you rename the file.

In [ ]:
# ============================================
# Cell 2 — Configuration
# ============================================
HASH_DB = "hashes.json"   # name of the file that stores all hashes

print(f"Hash database will be stored as: {HASH_DB}")

## 3. Compute Hash

This is the core function. It reads a file in small 4096-byte chunks and feeds each chunk into the SHA-256 algorithm.

**Why chunks?** Some log files can be gigabytes large. Reading in chunks means we never load the whole file into memory at once.

In [ ]:
# ============================================
# Cell 3 — Compute SHA-256 hash of a file
# ============================================
def compute_hash(filepath):
    """
    Read a file in 4096-byte chunks and return its SHA-256 fingerprint.
    Returns a 64-character hexadecimal string.
    """
    sha256 = hashlib.sha256()                         # create a SHA-256 hasher object
    with open(filepath, "rb") as f:                   # open in binary mode (rb)
        for chunk in iter(lambda: f.read(4096), b""): # read 4096 bytes at a time
            sha256.update(chunk)                       # feed chunk into the hasher
    return sha256.hexdigest()                          # return the final 64-char hash


# ── Quick demo ──
# Create a tiny test file and show its hash
with open("demo.txt", "w") as f:
    f.write("Hello, world!")

h = compute_hash("demo.txt")
print(f"File: demo.txt")
print(f"Hash: {h}")
print(f"Length: {len(h)} characters (always 64 for SHA-256)")

## 4. Load and Save the Database

The database is a simple JSON file. JSON is like a Python dictionary saved to disk — keys are file paths, values are hash + metadata.

In [ ]:
# ============================================
# Cell 4 — Load / save hash database
# ============================================
def load_db():
    """Load existing hash database, or return empty dict if none exists."""
    if not os.path.exists(HASH_DB):
        return {}
    with open(HASH_DB, "r") as f:
        return json.load(f)

def save_db(data):
    """Save the hash database to hashes.json."""
    with open(HASH_DB, "w") as f:
        json.dump(data, f, indent=2)  # indent=2 makes the JSON human-readable

print("✅ Database functions defined")

## 5. Helper: Collect Files

This helper accepts either a single file path or a folder path, and always returns a flat list of file paths to process.

In [ ]:
# ============================================
# Cell 5 — Helper: collect all file paths
# ============================================
def collect_files(path):
    """
    Return a list of file paths from input.
    - Single file  →  returns [path]
    - Directory    →  returns all files inside (recursively)
    """
    if os.path.isfile(path):
        return [path]
    elif os.path.isdir(path):
        all_files = []
        for root, dirs, files in os.walk(path):  # os.walk goes into subfolders too
            for filename in files:
                all_files.append(os.path.join(root, filename))
        return all_files
    else:
        print(f"[ERROR] Path not found: {path}")
        return []

print("✅ collect_files() defined")

## 6. Init Command

Scans a file or folder, computes a hash for each file, and saves everything to `hashes.json`. This creates the **trusted baseline**.

In [ ]:
# ============================================
# Cell 6 — init() command
# ============================================
def init(path):
    """Scan a file or folder and store hashes for all files."""
    files = collect_files(path)
    if not files:
        return

    db = load_db()
    new_count = 0
    updated_count = 0

    for filepath in files:
        abs_path = os.path.abspath(filepath)      # always store absolute paths
        file_hash = compute_hash(abs_path)
        is_new = abs_path not in db

        db[abs_path] = {
            "hash": file_hash,
            "last_updated": datetime.now().isoformat(),
            "size_bytes": os.path.getsize(abs_path)
        }

        if is_new:
            new_count += 1
        else:
            updated_count += 1

    save_db(db)
    print(f"\n✅ Hashes stored successfully.")
    print(f"   New files recorded : {new_count}")
    print(f"   Re-hashed files    : {updated_count}")
    print(f"   Database saved to  : {HASH_DB}\n")

print("✅ init() defined")

## 7. Check Command

Recomputes hashes and compares them against the stored baseline. Reports Unmodified ✅, Modified ⚠️, or New 🆕.

In [ ]:
# ============================================
# Cell 7 — check() command
# ============================================
def check(path):
    """Compare current file hashes against the stored baseline."""
    db = load_db()

    if not db:
        print("\n[ERROR] No hash database found. Run init() first.\n")
        return

    files = collect_files(path)
    if not files:
        return

    print(f"\n{'─'*55}")
    print(f"  Integrity Check Report  —  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'─'*55}")

    unmodified = 0
    modified = 0
    new_files = 0

    for filepath in files:
        abs_path = os.path.abspath(filepath)
        display  = os.path.basename(abs_path)

        if abs_path not in db:
            print(f"  🆕  {display}")
            print(f"       Status : NEW FILE (not in database)")
            new_files += 1
        else:
            current_hash = compute_hash(abs_path)
            stored_hash  = db[abs_path]["hash"]

            if current_hash == stored_hash:
                print(f"  ✅  {display}")
                print(f"       Status : Unmodified")
                unmodified += 1
            else:
                print(f"  ⚠️   {display}")
                print(f"       Status : MODIFIED — possible tampering detected!")
                print(f"       Stored hash  : {stored_hash[:32]}...")
                print(f"       Current hash : {current_hash[:32]}...")
                modified += 1

    print(f"{'─'*55}")
    print(f"  Summary: {unmodified} unmodified  |  {modified} modified  |  {new_files} new")
    print(f"{'─'*55}\n")

print("✅ check() defined")

## 8. Update Command

Use this when you've **intentionally** changed a file and want to accept its new hash as the new trusted baseline.

In [ ]:
# ============================================
# Cell 8 — update() command
# ============================================
def update(path):
    """Update the stored hash for a specific file."""
    db = load_db()

    if not db:
        print("\n[ERROR] No hash database found. Run init() first.\n")
        return

    if not os.path.isfile(path):
        print(f"\n[ERROR] Not a file: {path}\n")
        return

    abs_path = os.path.abspath(path)
    new_hash = compute_hash(abs_path)

    db[abs_path] = {
        "hash": new_hash,
        "last_updated": datetime.now().isoformat(),
        "size_bytes": os.path.getsize(abs_path)
    }

    save_db(db)
    print(f"\n✅ Hash updated for: {abs_path}")
    print(f"   New hash : {new_hash}\n")

print("✅ update() defined")

## 9. List Command

Shows all files currently tracked in the hash database.

In [ ]:
# ============================================
# Cell 9 — list_files() command
# ============================================
def list_files():
    """Show all files currently tracked in the hash database."""
    db = load_db()

    if not db:
        print("\n[INFO] No hash database found. Run init() first.\n")
        return

    print(f"\n{'─'*55}")
    print(f"  Tracked Files ({len(db)} total)")
    print(f"{'─'*55}")

    for filepath, data in db.items():
        print(f"  📄 {os.path.basename(filepath)}")
        print(f"     Path    : {filepath}")
        print(f"     Hash    : {data['hash'][:32]}...")
        print(f"     Updated : {data['last_updated']}")
        print(f"     Size    : {data['size_bytes']} bytes")
        print()

    print(f"{'─'*55}\n")

print("✅ list_files() defined")

## 10. Full Demo — Run Everything

This cell demonstrates the complete workflow:
1. Create sample log files
2. Init (store hashes)
3. Check (all clean)
4. Tamper with a file
5. Check again (tampering detected!)
6. Update the hash
7. Check again (clean)

In [ ]:
# ============================================
# Cell 10 — Full Demo
# ============================================

# --- Step 1: Create sample log files ---
os.makedirs("demo_logs", exist_ok=True)

with open("demo_logs/app.log", "w") as f:
    f.write("2024-01-15 10:00:00 INFO  Server started\n")
    f.write("2024-01-15 10:01:00 INFO  User admin logged in\n")

with open("demo_logs/auth.log", "w") as f:
    f.write("2024-01-15 10:00:05 INFO  Auth service ready\n")
    f.write("2024-01-15 10:01:00 INFO  Login success: admin\n")

print("📁 Created demo_logs/app.log and demo_logs/auth.log")
print("─" * 55)

# --- Step 2: Init — store baseline hashes ---
print("\n▶ STEP 2: Init (store hashes)")
init("demo_logs")

# --- Step 3: Check — should be all clean ---
print("\n▶ STEP 3: Check (should be Unmodified)")
check("demo_logs")

# --- Step 4: Tamper with app.log ---
print("\n💀 TAMPERING with demo_logs/app.log...")
with open("demo_logs/app.log", "a") as f:
    f.write("2024-01-15 11:00:00 ERROR  ** LOG ENTRY INJECTED BY ATTACKER **\n")
print("   Malicious line added to app.log")

# --- Step 5: Check — should detect tampering ---
print("\n▶ STEP 5: Check (should detect MODIFIED)")
check("demo_logs")

# --- Step 6: Update the hash ---
print("\n▶ STEP 6: Update hash for app.log")
update("demo_logs/app.log")

# --- Step 7: Final check — all clean again ---
print("\n▶ STEP 7: Final check (should be Unmodified again)")
check("demo_logs")

## 11. Save as a .py script

Run this cell to save the complete tool as `integrity_checker.py` — this is the file you push to GitHub.

In [ ]:
# ============================================
# Cell 11 — Download the .py file
# ============================================
# If you're in Colab, this downloads integrity_checker.py to your computer
try:
    from google.colab import files
    files.download('integrity_checker.py')
    print("✅ Download started!")
except ImportError:
    print("Not running in Colab — file is already saved locally.")

---

## What I learned

- **SHA-256 hashing** — how cryptographic fingerprints work and why even 1 character change produces a completely different hash
- **File I/O in Python** — reading files in binary mode with chunks for memory efficiency  
- **JSON storage** — saving and loading structured data to disk
- **Command-line tools** — building a multi-command CLI tool with `sys.argv`
- **Security concepts** — file integrity monitoring (FIM) as used in real security products like Tripwire and OSSEC
